In [1]:
import sys                      # Módulo para inspeccionar el intérprete en ejecución
print(sys.executable)           # Debe apuntar a .../.venv/bin/python; si no, cambia el kernel

/home/benmp/dev/pln/nlp-text-normalization-spanish/.venv/bin/python


In [2]:
from nltk import ngrams
from collections import defaultdict, Counter
import random
import re
import numpy as np

corpus = [
    "el gato está en el tejado",
    "el perro está en el jardín",
    "el gato come en el jardín",
    "el perro come en el tejado"
]

In [3]:
def preprocesar_para_ngrama(texto):
    texto = texto.lower().strip()
    texto = re.sub(r'[^\w\s]', '', texto)  # solo quitar puntuación
    # NO lematizar agresivamente, NO quitar stopwords
    # porque el contexto secuencial importa
    return texto.split()

In [ ]:
def entrenar_modelo(corpus, n=2, suavizado=1):
    modelo = defaultdict(Counter)
    vocabulario = set()

    for frase in corpus:
        tokens = preprocesar_para_ngrama(frase)
        vocabulario.update(tokens)
        for grama in ngrams(tokens, n, pad_left=True, pad_right=True,
                           left_pad_symbol="<s>", right_pad_symbol="</s>"):
            contexto = grama[:-1]
            siguiente = grama[-1]
            modelo[contexto][siguiente] += 1

    # El suavizado se aplica aquí, al momento de calcular las probabilidades
    # y se guarda en el modelo, no se pasa como parámetro después
    probabilidades = defaultdict(dict)
    for contexto, conteos in modelo.items():
        total = sum(conteos.values()) + suavizado * len(vocabulario)
        for palabra in vocabulario:
            probabilidades[contexto][palabra] = (conteos[palabra] + suavizado) / total

    return probabilidades, vocabulario


In [ ]:
# Ahora predecir y calcular perplejidad solo consumen probabilidades
def predecir_siguiente(probabilidades, contexto):
    if contexto not in probabilidades:
        return []
    return sorted(probabilidades[contexto].items(), key=lambda x: x[1], reverse=True)[:6]



# --- Comparar distintos valores de N ---
for n in [2, 3]:
    print(f"\n{'='*40}")
    print(f"Modelo con N={n}")
    modelo, vocab = entrenar_modelo(corpus, n=n)

    contexto = ("el", ) if n == 2 else ("el", "gato")[-n+1:]
    resultado = predecir_siguiente(modelo, contexto)

    print(f"Contexto: {contexto}")
    for palabra, prob in resultado:
        print(f"  P({palabra} | contexto) = {prob:.3f}")


Modelo con N=2
Contexto: ('el',)
  P(gato | contexto) = 0.188
  P(jardín | contexto) = 0.188
  P(tejado | contexto) = 0.188
  P(perro | contexto) = 0.188
  P(come | contexto) = 0.062
  P(está | contexto) = 0.062

Modelo con N=3
Contexto: ('el', 'gato')
  P(come | contexto) = 0.200
  P(está | contexto) = 0.200
  P(gato | contexto) = 0.100
  P(jardín | contexto) = 0.100
  P(tejado | contexto) = 0.100
  P(perro | contexto) = 0.100
